d13_s3_a2.ipynb

In [ ]:
import numpy as np
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# Step 1: Load Dataset
d = load_breast_cancer()

X = d.data
y = d.target


# Step 2: Split Dataset
Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=0
)


# Step 3: Train Model and Save Pipeline
def train_and_save(path, seed=0):

    d = load_breast_cancer()

    X = d.data
    y = d.target

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000))
    ])

    model.fit(X_train, y_train)

    joblib.dump(model, path)

    return model


# Step 4: Load Saved Model
def load_model(path):

    model = joblib.load(path)

    return model


# Step 5: Predict Positive Class Probabilities
def predict_proba_positive(model, X):

    probabilities = model.predict_proba(X)[:, 1]

    return np.asarray(probabilities, dtype=float)


# Step 6: Check Save and Load Consistency
def roundtrip_consistent(model, path, X):

    joblib.dump(model, path)

    loaded_model = joblib.load(path)

    original_predictions = model.predict(X)

    loaded_predictions = loaded_model.predict(X)

    return bool(np.array_equal(
        original_predictions,
        loaded_predictions
    ))


# Step 7: Train and Save Model
m = train_and_save("model.joblib", seed=0)


# Step 8: Load Model for Serving
srv = load_model("model.joblib")


# Step 9: Compare Predictions
print(
    "Predictions Match:",
    (srv.predict(Xte) == m.predict(Xte)).all()
)


# Step 10: Predict Probabilities
print(
    "Positive Class Probabilities:",
    predict_proba_positive(srv, Xte[:3]).round(3)
)


# Step 11: Check Roundtrip Consistency
print(
    "Roundtrip Consistent:",
    roundtrip_consistent(m, "rt.joblib", Xte)
)


# Step 12: Check Accuracy
predictions = srv.predict(Xte)

accuracy = accuracy_score(yte, predictions)

print("Test Accuracy:", round(float(accuracy), 3))


# Step 13: Verify Pipeline Steps
print("Pipeline Steps:", srv.named_steps.keys())

print("Saved Model Type:", type(srv).__name__)